# CTC OCR Training for Upper Plates (Special Characters)
- Fine-tune จากโมเดลเดิม (`upper_ctc_best.pt`) ที่เทรนกับตัวอักษรปกติแล้ว (accuracy ~90%)
- ใช้ dataset ใหม่: `upper_train_with_special` ที่รวมข้อมูล real + synthetic special plates
- **Character set ขยาย**: เพิ่มสระ (ิ ี ึ ื ั ุ ู เ แ โ ใ ไ ฤ ฦ ะ า ำ) และวรรณยุกต์ (่ ้ ๊ ๋) ครบทุกตัว
- **Fine-tuning strategy**: โหลด CNN+LSTM weights จากโมเดลเดิม แต่ **reset classifier head** เพื่อรองรับ character set ใหม่
- Test sets: `upper_test` (real), `plate_upper_synth_test` (synthetic normal), `plate_upper_synth_special_test` (synthetic special)

**Colab mode:**
- Load pretrained weights from `/content/drive/MyDrive/ALPRV2/upper_ctc_best.pt`
- Mount Google Drive, unzip datasets
- Save new checkpoint to `/content/drive/MyDrive/ALPR_Project/alpr_ctc_upper_special`

In [5]:
# Cell 1: Dataset paths (Local laptop: use existing folders; Colab: optional zip/unzip)
import os, zipfile, shutil
from pathlib import Path

def is_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

IS_COLAB = is_colab()
print('IS_COLAB:', IS_COLAB)

def find_repo_root(start: Path) -> Path:
    """Walk upwards to find a folder containing train_ocr/data/."""
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / 'train_ocr' / 'data').exists():
            return p
    return start

def resolve_local_data_root() -> str:
    repo_root = find_repo_root(Path.cwd())
    return str((repo_root / 'train_ocr' / 'data').resolve())

def unzip_to(zip_path: str, out_dir: str):
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f'Zip not found: {zip_path}')
    if os.path.exists(out_dir) and os.path.isdir(out_dir) and len(os.listdir(out_dir)) > 0:
        print(f'Skip (already exists): {out_dir}')
        return
    if os.path.exists(out_dir):
        shutil.rmtree(out_dir)
    os.makedirs(out_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(out_dir)
    print(f'Unzipped: {zip_path} -> {out_dir}')

def resolve_dataset_dir(root_dir: str) -> str:
    # Supports both layouts:
    # 1) root_dir/labels.csv + root_dir/data/...
    # 2) root_dir/<subdir>/labels.csv + ...
    if os.path.exists(os.path.join(root_dir, 'labels.csv')):
        return root_dir
    for d in os.listdir(root_dir):
        sd = os.path.join(root_dir, d)
        if os.path.isdir(sd) and os.path.exists(os.path.join(sd, 'labels.csv')):
            return sd
    raise FileNotFoundError(f'Could not find labels.csv under: {root_dir}')

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # Pretrained weights from previous training
    PRETRAINED_WEIGHTS = '/content/drive/MyDrive/ALPRV2/upper_ctc_best.pt'

    # New datasets
    ZIP_TRAIN = '/content/drive/MyDrive/ALPRV2/upper_train_with_special.zip'
    ZIP_TEST = '/content/drive/MyDrive/ALPRV2/upper_test.zip'
    ZIP_TEST_SYN = '/content/drive/MyDrive/ALPRV2/plate_upper_synth_test.zip'
    ZIP_TEST_SYN_SPECIAL = '/content/drive/MyDrive/ALPRV2/plate_upper_synth_special_test.zip'

    OUT_ROOT = '/content/datasets'
    RAW_TRAIN_DIR = os.path.join(OUT_ROOT, 'upper_train_with_special')
    RAW_TEST_DIR = os.path.join(OUT_ROOT, 'upper_test')
    RAW_TEST_SYN_DIR = os.path.join(OUT_ROOT, 'plate_upper_synth_test')
    RAW_TEST_SYN_SPECIAL_DIR = os.path.join(OUT_ROOT, 'plate_upper_synth_special_test')
    os.makedirs(OUT_ROOT, exist_ok=True)

    unzip_to(ZIP_TRAIN, RAW_TRAIN_DIR)
    unzip_to(ZIP_TEST, RAW_TEST_DIR)
    unzip_to(ZIP_TEST_SYN, RAW_TEST_SYN_DIR)
    unzip_to(ZIP_TEST_SYN_SPECIAL, RAW_TEST_SYN_SPECIAL_DIR)

    TRAIN_DIR = resolve_dataset_dir(RAW_TRAIN_DIR)
    TEST_DIR = resolve_dataset_dir(RAW_TEST_DIR)
    TEST_SYN_DIR = resolve_dataset_dir(RAW_TEST_SYN_DIR)
    TEST_SYN_SPECIAL_DIR = resolve_dataset_dir(RAW_TEST_SYN_SPECIAL_DIR)

    DATA_ROOT_PATH = '/content/datasets'
    SAVE_DIR = '/content/drive/MyDrive/ALPR_Project/alpr_ctc_upper_special'
else:
    PRETRAINED_WEIGHTS = str((Path.cwd() / 'weights' / 'upper_ctc_best.pt').resolve())
    DATA_ROOT_PATH = resolve_local_data_root()
    SAVE_DIR = str((Path.cwd() / 'weights').resolve())

print('PRETRAINED_WEIGHTS:', PRETRAINED_WEIGHTS if IS_COLAB else 'N/A (local training from scratch)')
print('DATA_ROOT_PATH    :', DATA_ROOT_PATH)
print('SAVE_DIR          :', SAVE_DIR)

IS_COLAB: True
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Skip (already exists): /content/datasets/upper_train_with_special
Skip (already exists): /content/datasets/upper_test
Skip (already exists): /content/datasets/plate_upper_synth_test
Skip (already exists): /content/datasets/plate_upper_synth_special_test
PRETRAINED_WEIGHTS: /content/drive/MyDrive/ALPRV2/upper_ctc_best.pt
DATA_ROOT_PATH    : /content/datasets
SAVE_DIR          : /content/drive/MyDrive/ALPR_Project/alpr_ctc_upper_special


In [6]:
from __future__ import annotations

from pathlib import Path
import math
import random
import json
import pandas as pd
import numpy as np
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DATA_ROOT = Path(DATA_ROOT_PATH)

# Updated dataset paths for special character training
if IS_COLAB:
    TRAIN_LABELS = DATA_ROOT / "upper_train_with_special" / "upper_train_with_special" / "labels.csv"
    TRAIN_IMAGES = DATA_ROOT / "upper_train_with_special" / "upper_train_with_special" / "data"
    TEST_REAL_LABELS = DATA_ROOT / "upper_test" / "upper_test" / "labels.csv"
    TEST_REAL_IMAGES = DATA_ROOT / "upper_test" / "upper_test" / "data"
    TEST_SYN_LABELS = DATA_ROOT / "plate_upper_synth_test" / "plate_upper_synth_test" / "labels.csv"
    TEST_SYN_IMAGES = DATA_ROOT / "plate_upper_synth_test" / "plate_upper_synth_test" / "data"
    TEST_SYN_SPECIAL_LABELS = DATA_ROOT / "plate_upper_synth_special_test" / "plate_upper_synth_special_test" / "labels.csv"
    TEST_SYN_SPECIAL_IMAGES = DATA_ROOT / "plate_upper_synth_special_test" / "plate_upper_synth_special_test" / "data"
else:
    TRAIN_LABELS = DATA_ROOT / "upper_train_with_special" / "upper_train_with_special" / "labels.csv"
    TRAIN_IMAGES = DATA_ROOT / "upper_train_with_special" / "upper_train_with_special" / "data"
    TEST_REAL_LABELS = DATA_ROOT / "upper_test" / "upper_test" / "labels.csv"
    TEST_REAL_IMAGES = DATA_ROOT / "upper_test" / "upper_test" / "data"
    TEST_SYN_LABELS = DATA_ROOT / "plate_upper_synth_test" / "plate_upper_synth_test" / "labels.csv"
    TEST_SYN_IMAGES = DATA_ROOT / "plate_upper_synth_test" / "plate_upper_synth_test" / "data"
    TEST_SYN_SPECIAL_LABELS = DATA_ROOT / "plate_upper_synth_special_test" / "plate_upper_synth_special_test" / "labels.csv"
    TEST_SYN_SPECIAL_IMAGES = DATA_ROOT / "plate_upper_synth_special_test" / "plate_upper_synth_special_test" / "data"

IMG_HEIGHT = 32
IMG_WIDTH = 128
BATCH_SIZE = 128
EPOCHS = 15  # Fine-tuning typically needs fewer epochs
LR = 1e-4  # Lower learning rate for fine-tuning
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 8
TRAIN_FRAC = 0.9
GRAD_CLIP = 5.0

if 'IS_COLAB' in globals() and IS_COLAB:
    BATCH_SIZE = 128
    NUM_WORKERS = 2
    print("Colab overrides: BATCH_SIZE=128, NUM_WORKERS=2")

CHECKPOINT_PATH = Path(SAVE_DIR) / "upper_ctc_special_best.pt"
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Colab overrides: BATCH_SIZE=128, NUM_WORKERS=2
Device: cuda


In [7]:
def load_labels(csv_path: Path, images_dir: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path, encoding='utf-8-sig')
    if "label" not in df.columns or "filename" not in df.columns:
        raise ValueError("labels.csv must contain 'filename' and 'label' columns")
    df["filename"] = df["filename"].astype(str).str.strip()
    df["label"] = df["label"].astype(str).str.strip()
    df["full_path"] = df["filename"].apply(lambda x: images_dir / x)
    missing = (~df["full_path"].apply(Path.exists)).sum()
    if missing:
        print(f"Warning: {missing} files missing for {csv_path}")
    return df

df_full = load_labels(TRAIN_LABELS, TRAIN_IMAGES)
df_full = df_full.sample(frac=1, random_state=SEED).reset_index(drop=True)
cut = int(len(df_full) * TRAIN_FRAC)
df_train = df_full.iloc[:cut].copy()
df_val = df_full.iloc[cut:].copy()
print("Train/Val split:", len(df_train), len(df_val))

df_test_real = load_labels(TEST_REAL_LABELS, TEST_REAL_IMAGES)
df_test_synth = load_labels(TEST_SYN_LABELS, TEST_SYN_IMAGES)
df_test_synth_special = load_labels(TEST_SYN_SPECIAL_LABELS, TEST_SYN_SPECIAL_IMAGES)
print("Test sets:", len(df_test_real), len(df_test_synth), len(df_test_synth_special))

/tmp/ipython-input-1441274321.py:2: DtypeWarning: Columns (3,5,6,7,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path, encoding='utf-8-sig')


Train/Val split: 187349 20817
Test sets: 2042 5000 5000


In [8]:
# Extended character set with ALL Thai vowels, tone marks, and special characters
DIGITS = "0123456789"
THAI_CONSONANTS = "กขคฆงจฉชซฌญฎฏฐฑฒณดตถทธนบปผฝพฟภมยรลวศษสหฬอฮ"  # exclude obsolete ฃ ฅ
THAI_VOWELS = "ิีึืัุูเแโใไฤฦะาำ"  # all Thai vowels including sara am
TONE_MARKS = "่้๊๋"  # all 4 tone marks
SPECIAL_MARKS = "็์ๆฯ"  # mai tai khu, thanthakhat, repetition, abbreviation
SPACE = " "  # allow space for special plates

ALLOWED_CHARS = list(DIGITS + THAI_CONSONANTS + THAI_VOWELS + TONE_MARKS + SPECIAL_MARKS + SPACE)

def filter_unsupported(df: pd.DataFrame, label_col: str = "label") -> pd.DataFrame:
    def has_unsupported(s: str) -> bool:
        return any(ch not in ALLOWED_CHARS for ch in str(s))
    mask = df[label_col].apply(has_unsupported)
    if mask.any():
        print(f"Filtered {mask.sum()} rows with unsupported chars in {label_col}")
        # Show some examples
        examples = df[mask][label_col].head(5).tolist()
        print(f"Examples: {examples}")
    return df[~mask].copy()

df_train = filter_unsupported(df_train)
df_val = filter_unsupported(df_val)
df_test_real = filter_unsupported(df_test_real)
df_test_synth = filter_unsupported(df_test_synth)
df_test_synth_special = filter_unsupported(df_test_synth_special)

def build_charset_fixed(labels: list[str]) -> tuple[list[str], dict[str, int]]:
    unknown = set()
    for text in labels:
        for ch in str(text):
            if ch not in ALLOWED_CHARS:
                unknown.add(ch)
    if unknown:
        preview = " ".join(sorted(list(unknown))[:30])
        print(f"Warning: found unsupported chars (ignored in charset): {preview}")
    idx_to_char = ["<BLANK>"] + ALLOWED_CHARS
    char_to_idx = {c: i for i, c in enumerate(idx_to_char)}
    return idx_to_char, char_to_idx

all_labels = pd.concat([
    df_train["label"],
    df_val["label"],
    df_test_real["label"],
    df_test_synth["label"],
    df_test_synth_special["label"],
], ignore_index=True).tolist()

idx_to_char, char_to_idx = build_charset_fixed(all_labels)
print("Charset size (incl. blank):", len(idx_to_char))
print("Charset breakdown:")
print(f"  - Digits: {DIGITS}")
print(f"  - Consonants: {THAI_CONSONANTS}")
print(f"  - Vowels: {THAI_VOWELS}")
print(f"  - Tone marks: {TONE_MARKS}")
print(f"  - Special marks: {SPECIAL_MARKS}")
print(f"  - Space: (included)")
print(f"Total characters (excluding blank): {len(ALLOWED_CHARS)}")

Charset size (incl. blank): 79
Charset breakdown:
  - Digits: 0123456789
  - Consonants: กขคฆงจฉชซฌญฎฏฐฑฒณดตถทธนบปผฝพฟภมยรลวศษสหฬอฮ
  - Vowels: ิีึืัุูเแโใไฤฦะาำ
  - Tone marks: ่้๊๋
  - Special marks: ็์ๆฯ
  - Space: (included)
Total characters (excluding blank): 78


In [9]:
class OCRDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        img = Image.open(row["full_path"]).convert("L")
        # Assert size already 128x32 to avoid unintended resizing
        assert img.size == (IMG_WIDTH, IMG_HEIGHT), f"Unexpected size {img.size} for {row['full_path']}"
        img = self.transform(img)
        label_text = row["label"]
        target = torch.tensor([char_to_idx[c] for c in label_text], dtype=torch.long)
        return img, target, label_text

train_transform = T.Compose([
    T.RandomApply([T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.02)], p=0.7),
    T.RandomApply([T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.2))], p=0.2),
    T.RandomAffine(degrees=2, translate=(0.02, 0.05), scale=(0.95, 1.05), shear=1, fill=0),
    T.ToTensor(),
    T.Normalize((0.5,), (0.5,)),
])

eval_transform = T.Compose([
    T.ToTensor(),
    T.Normalize((0.5,), (0.5,)),
])

def collate_fn(batch):
    imgs, targets, texts = zip(*batch)
    imgs = torch.stack(imgs)
    target_lengths = torch.tensor([t.numel() for t in targets], dtype=torch.long)
    targets = torch.cat(targets)
    return imgs, targets, target_lengths, list(texts)

train_ds = OCRDataset(df_train, train_transform)
val_ds = OCRDataset(df_val, eval_transform)
test_real_ds = OCRDataset(df_test_real, eval_transform)
test_synth_ds = OCRDataset(df_test_synth, eval_transform)
test_synth_special_ds = OCRDataset(df_test_synth_special, eval_transform)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate_fn,
    prefetch_factor=4,
    persistent_workers=True
)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn, prefetch_factor=4, persistent_workers=True)
test_real_loader = DataLoader(test_real_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn, prefetch_factor=4, persistent_workers=True)
test_synth_loader = DataLoader(test_synth_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn, prefetch_factor=4, persistent_workers=True)
test_synth_special_loader = DataLoader(test_synth_special_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn, prefetch_factor=4, persistent_workers=True)

print("DataLoaders created successfully")
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

DataLoaders created successfully
Train batches: 1464, Val batches: 163


In [10]:
class CRNN(nn.Module):
    def __init__(self, num_classes: int, hidden: int = 256):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.Conv2d(256, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.MaxPool2d((2, 1), (2, 1)),
            nn.Conv2d(256, 512, 3, 1, 1), nn.BatchNorm2d(512), nn.ReLU(True),
            nn.MaxPool2d((2, 1), (2, 1)),
        )
        self.rnn = nn.LSTM(512 * (IMG_HEIGHT // 16), hidden, num_layers=2, batch_first=True, bidirectional=True)
        self.classifier = nn.Linear(hidden * 2, num_classes)

    @staticmethod
    def seq_len_from_width(width: int) -> int:
        # width reduced by factor 4 from two MaxPool2d(2,2) layers
        return width // 4

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feats = self.cnn(x)
        b, c, h, w = feats.size()
        feats = feats.permute(0, 3, 1, 2).contiguous()
        feats = feats.view(b, w, c * h)
        rnn_out, _ = self.rnn(feats)
        logits = self.classifier(rnn_out)  # (B, T, C)
        return logits.permute(1, 0, 2)  # (T, B, C) for CTC

In [11]:
def greedy_decode(logits: torch.Tensor) -> list[str]:
    # logits: (T, B, C)
    probs = logits.softmax(2)
    indices = probs.argmax(2).permute(1, 0)  # (B, T)
    texts = []
    for seq in indices:
        prev = None
        chars = []
        for idx in seq.tolist():
            if idx != 0 and idx != prev:
                chars.append(idx_to_char[idx])
            prev = idx
        texts.append("".join(chars))
    return texts

def step(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    total_loss = 0.0
    total_correct = 0
    total_count = 0
    model.train(is_train)
    for imgs, targets, target_lengths, texts in loader:
        imgs = imgs.to(device)
        targets = targets.to(device)
        target_lengths = target_lengths.to(device)
        logits = model(imgs)
        log_probs = logits.log_softmax(2)
        input_lengths = torch.full((imgs.size(0),), logits.size(0), dtype=torch.long, device=device)
        loss = criterion(log_probs, targets, input_lengths, target_lengths)
        if is_train:
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        preds = greedy_decode(log_probs.detach())
        total_correct += sum(p == t for p, t in zip(preds, texts))
        total_count += len(texts)
    avg_loss = total_loss / max(1, total_count)
    acc = total_correct / max(1, total_count)
    return avg_loss, acc

def evaluate(model, loader, criterion, tag: str):
    with torch.no_grad():
        loss, acc = step(model, loader, criterion, optimizer=None)
    print(f"[{tag}] loss={loss:.4f} acc={acc:.4f}")
    return loss, acc

In [12]:
# Load pretrained model and transfer weights (with head reset)
def load_pretrained_and_reset_head(pretrained_path: str, new_num_classes: int, device: torch.device):
    """
    Load pretrained model, transfer CNN+RNN weights, but reset classifier head for new character set.
    """
    if not os.path.exists(pretrained_path):
        print(f"⚠️ Pretrained weights not found: {pretrained_path}")
        print("Training from scratch...")
        return CRNN(num_classes=new_num_classes).to(device), None

    print(f"Loading pretrained weights from: {pretrained_path}")
    ckpt = torch.load(pretrained_path, map_location=device)
    old_num_classes = len(ckpt["idx_to_char"])

    # Create old model with old character set
    old_model = CRNN(num_classes=old_num_classes)
    old_model.load_state_dict(ckpt["model_state"])

    # Create new model with new character set
    new_model = CRNN(num_classes=new_num_classes).to(device)

    # Transfer CNN and RNN weights (feature extractor)
    new_state = new_model.state_dict()
    old_state = old_model.state_dict()

    transferred = 0
    skipped = 0
    for name, param in old_state.items():
        # Transfer all except classifier head
        if "classifier" not in name:
            if name in new_state and new_state[name].shape == param.shape:
                new_state[name] = param
                transferred += 1
            else:
                print(f"⚠️ Shape mismatch or missing: {name}")
                skipped += 1
        else:
            print(f"🔄 Reset (new charset): {name}")
            skipped += 1

    new_model.load_state_dict(new_state)

    print(f"✓ Transferred {transferred} layers, reset {skipped} layers (classifier head)")
    print(f"Old charset size: {old_num_classes}, New charset size: {new_num_classes}")

    return new_model, ckpt["idx_to_char"]

In [13]:
# Initialize model with pretrained weights (fine-tuning)
if IS_COLAB and os.path.exists(PRETRAINED_WEIGHTS):
    model, old_charset = load_pretrained_and_reset_head(PRETRAINED_WEIGHTS, len(idx_to_char), device)
    print("\n🎯 Fine-tuning mode: Using pretrained CNN+RNN, training new classifier head")
else:
    model = CRNN(num_classes=len(idx_to_char)).to(device)
    print("\n🆕 Training from scratch")

criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=LR, steps_per_epoch=len(train_loader), epochs=EPOCHS)

best_val_acc = 0.0
for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_count = 0
    for step_idx, (imgs, targets, target_lengths, texts) in enumerate(tqdm(train_loader, desc="train", leave=False), start=1):
        imgs = imgs.to(device)
        targets = targets.to(device)
        target_lengths = target_lengths.to(device)
        logits = model(imgs)
        log_probs = logits.log_softmax(2)
        input_lengths = torch.full((imgs.size(0),), logits.size(0), dtype=torch.long, device=device)
        loss = criterion(log_probs, targets, input_lengths, target_lengths)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item() * imgs.size(0)
        preds = greedy_decode(log_probs.detach())
        total_correct += sum(p == t for p, t in zip(preds, texts))
        total_count += len(texts)
        if step_idx % 200 == 0:
            running_loss = total_loss / max(1, total_count)
            running_acc = total_correct / max(1, total_count)
            print(f"  step {step_idx}: running loss={running_loss:.4f} acc={running_acc:.4f}")
    train_loss = total_loss / max(1, total_count)
    train_acc = total_correct / max(1, total_count)
    print(f"[train] loss={train_loss:.4f} acc={train_acc:.4f}")

    val_loss, val_acc = evaluate(model, val_loader, criterion, tag="val")
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "model_state": model.state_dict(),
            "idx_to_char": idx_to_char,
            "char_to_idx": char_to_idx,
            "config": {
                "IMG_HEIGHT": IMG_HEIGHT,
                "IMG_WIDTH": IMG_WIDTH,
            },
        }, CHECKPOINT_PATH)
        print(f"💾 Saved new best to {CHECKPOINT_PATH} (val_acc={val_acc:.4f})")

print(f"\n✅ Training complete! Best val accuracy: {best_val_acc:.4f}")

Loading pretrained weights from: /content/drive/MyDrive/ALPRV2/upper_ctc_best.pt
🔄 Reset (new charset): classifier.weight
🔄 Reset (new charset): classifier.bias
✓ Transferred 51 layers, reset 2 layers (classifier head)
Old charset size: 53, New charset size: 79

🎯 Fine-tuning mode: Using pretrained CNN+RNN, training new classifier head

Epoch 1/15


train:   0%|          | 0/1464 [00:00<?, ?it/s]

  step 200: running loss=12.8510 acc=0.0000
  step 400: running loss=8.6145 acc=0.0000
  step 600: running loss=7.0398 acc=0.0000
  step 800: running loss=6.1953 acc=0.0000
  step 1000: running loss=5.6463 acc=0.0000
  step 1200: running loss=5.2359 acc=0.0000
  step 1400: running loss=4.8847 acc=0.0000
[train] loss=4.7784 acc=0.0000
[val] loss=2.3201 acc=0.0000

Epoch 2/15


train:   0%|          | 0/1464 [00:00<?, ?it/s]

  step 200: running loss=2.0587 acc=0.0096
  step 400: running loss=1.8432 acc=0.0378
  step 600: running loss=1.6766 acc=0.0793
  step 800: running loss=1.5345 acc=0.1323
  step 1000: running loss=1.4067 acc=0.1961
  step 1200: running loss=1.2924 acc=0.2573
  step 1400: running loss=1.1890 acc=0.3100
[train] loss=1.1587 acc=0.3246
[val] loss=0.4588 acc=0.6577
💾 Saved new best to /content/drive/MyDrive/ALPR_Project/alpr_ctc_upper_special/upper_ctc_special_best.pt (val_acc=0.6577)

Epoch 3/15


...

  step 200: running loss=0.0174 acc=0.9608
  step 400: running loss=0.0181 acc=0.9604
  step 600: running loss=0.0175 acc=0.9613
  step 800: running loss=0.0175 acc=0.9614
  step 1000: running loss=0.0173 acc=0.9615
  step 1200: running loss=0.0175 acc=0.9611
  step 1400: running loss=0.0175 acc=0.9611
[train] loss=0.0175 acc=0.9612
[val] loss=0.0168 acc=0.9641
💾 Saved new best to /content/drive/MyDrive/ALPR_Project/alpr_ctc_upper_special/upper_ctc_special_best.pt (val_acc=0.9641)

Epoch 14/15


train:   0%|          | 0/1464 [00:00<?, ?it/s]

  step 200: running loss=0.0167 acc=0.9608
  step 400: running loss=0.0164 acc=0.9624
  step 600: running loss=0.0167 acc=0.9618
  step 800: running loss=0.0165 acc=0.9628
  step 1000: running loss=0.0164 acc=0.9628
  step 1200: running loss=0.0163 acc=0.9630
  step 1400: running loss=0.0163 acc=0.9630
[train] loss=0.0163 acc=0.9630
[val] loss=0.0165 acc=0.9645
💾 Saved new best to /content/drive/MyDrive/ALPR_Project/alpr_ctc_upper_special/upper_ctc_special_best.pt (val_acc=0.9645)

Epoch 15/15


train:   0%|          | 0/1464 [00:00<?, ?it/s]

  step 200: running loss=0.0168 acc=0.9627
  step 400: running loss=0.0162 acc=0.9641
  step 600: running loss=0.0161 acc=0.9642
  step 800: running loss=0.0160 acc=0.9645
  step 1000: running loss=0.0159 acc=0.9646
  step 1200: running loss=0.0159 acc=0.9645
  step 1400: running loss=0.0158 acc=0.9645
[train] loss=0.0158 acc=0.9644
[val] loss=0.0164 acc=0.9643

✅ Training complete! Best val accuracy: 0.9645


In [14]:
# Final evaluation on all test sets
def load_best(model_path: Path, device: torch.device):
    ckpt = torch.load(model_path, map_location=device)
    model = CRNN(num_classes=len(ckpt["idx_to_char"])).to(device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    return model, ckpt["idx_to_char"]

print("\n" + "="*60)
print("FINAL EVALUATION")
print("="*60)

best_model, _ = load_best(CHECKPOINT_PATH, device) if CHECKPOINT_PATH.exists() else (model, idx_to_char)

print("\n📊 Test Results:")
evaluate(best_model, test_real_loader, criterion, tag="test_real (real plates)")
evaluate(best_model, test_synth_loader, criterion, tag="test_synth (synthetic normal)")
evaluate(best_model, test_synth_special_loader, criterion, tag="test_synth_special (synthetic special)")

print("\n✅ All evaluations complete!")


FINAL EVALUATION

📊 Test Results:
[test_real (real plates)] loss=0.0485 acc=0.9594
[test_synth (synthetic normal)] loss=0.0011 acc=0.9990
[test_synth_special (synthetic special)] loss=0.0380 acc=0.9042

✅ All evaluations complete!
